# Personalized Item Recommendation (PIR) - Collaborative Filtering Baseline

## Overview
This notebook implements a collaborative filtering baseline for the PIR task using matrix factorization (TruncatedSVD). The pipeline:
- Loads transaction data and item metadata
- Splits data by time (Train: Jan-Sep 2025, Val: Oct 2025, Test: Nov 2025)
- Builds sparse user-item interaction matrices with binary implicit feedback
- Trains SVD decomposition and generates recommendations
- Evaluates performance with Precision@K, MRR, MAP@K, IoU metrics

**Kaggle-Friendly**: All code is self-contained, uses standard ML libraries, and handles path conventions automatically.

## 1. Kaggle Environment Setup

In [1]:
import os
import sys
import warnings
import json
from pathlib import Path

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
import numpy as np
np.random.seed(RANDOM_SEED)

# Detect Kaggle environment
IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    INPUT_DIR = Path('/kaggle/input')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    # Local environment
    INPUT_DIR = Path.cwd().parent.parent / 'data'  # Adjust as needed
    OUTPUT_DIR = Path.cwd() / 'outputs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Kaggle Environment: {IS_KAGGLE}")
print(f"Input Directory: {INPUT_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

Kaggle Environment: False
Input Directory: d:\CS116\data
Output Directory: d:\CS116\ProjectNumberOne\QuocKien\outputs


## 2. Imports and Dependency Checks

In [2]:
from dataclasses import dataclass
from typing import Dict, Iterable, List, Set, Sequence, Tuple
import json

import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

print("✓ All imports successful")
print(f"  - NumPy {np.__version__}")
print(f"  - Polars {pl.__version__}")
print(f"  - SciPy (sparse matrix support)")
print(f"  - scikit-learn (TruncatedSVD)")


✓ All imports successful
  - NumPy 2.4.4
  - Polars 1.40.1
  - SciPy (sparse matrix support)
  - scikit-learn (TruncatedSVD)


## 3. Configuration and Pipeline Settings

In [15]:
# Time split constants (year 2025)
TRAIN_MONTHS = {1, 2, 3, 4, 5, 6, 7, 8, 9}  # Jan-Sep
VAL_MONTHS = {10}  # Oct
TEST_MONTHS = {11}  # Nov

# Model hyperparameters
MODEL_CONFIG = {
    'n_components': 64,
    'random_state': 42,
    'n_iter': 100,
    'top_k': 10,  # Recommendation list size
    'batch_size': 1_000_000,  # Rows to process in parallel
}

# Data file names (Kaggle dataset specific)
TRANSACTION_FILE = 'transaction_full_2025.parquet'
ITEMS_FILENAMES = ['items.parquet', 'item_metadata.parquet', 'item_data.parquet']
SUBMISSION_FILE = 'submission.json'

print(f"✓ Configuration loaded")
print(f"  - Train months: {sorted(TRAIN_MONTHS)}")
print(f"  - Val month: {sorted(VAL_MONTHS)}")
print(f"  - Test month: {sorted(TEST_MONTHS)}")
print(f"  - SVD components: {MODEL_CONFIG['n_components']}")
print(f"  - Top-K recommendations: {MODEL_CONFIG['top_k']}")

✓ Configuration loaded
  - Train months: [1, 2, 3, 4, 5, 6, 7, 8, 9]
  - Val month: [10]
  - Test month: [11]
  - SVD components: 64
  - Top-K recommendations: 10


## 4. Data Loading and Schema Validation

In [18]:
def _detect_columns(df: pl.DataFrame) -> Tuple[str, str, str, str, str]:
    """Detect column names in transaction data for flexibility."""
    cols = set(df.columns)
    
    # Try common column name variations
    customer_col = next((c for c in cols if c.lower() in ['customer_id', 'customer', 'user_id', 'user']), None)
    item_col = next((c for c in cols if c.lower() in ['item_id', 'item', 'product_id', 'product']), None)
    date_col = next((c for c in cols if c.lower() in ['date', 'transaction_date', 'timestamp', 'updated_date', 'order_date']), None)
    event_col = next((c for c in cols if c.lower() in ['event_type', 'event', 'type']), None)
    qty_col = next((c for c in cols if c.lower() in ['quantity', 'qty', 'amount', 'price']), None)
    
    if not all([customer_col, item_col, date_col, event_col, qty_col]):
        missing = []
        if not customer_col: missing.append('customer_id')
        if not item_col: missing.append('item_id')
        if not date_col: missing.append('date')
        if not event_col: missing.append('event_type')
        if not qty_col: missing.append('quantity')
        raise KeyError(f"Missing required columns: {missing}. Found: {cols}")
    
    return customer_col, item_col, date_col, event_col, qty_col

def _format_month(date_val) -> Tuple[int, int]:
    """Extract year and month from date."""
    try:
        if isinstance(date_val, str):
            parts = date_val.split('-')
            return int(parts[0]), int(parts[1])
        else:
            return date_val.year, date_val.month
    except:
        return None, None

def load_items(items_path: Path) -> pl.DataFrame:
    """Load item metadata from parquet."""
    items = pl.scan_parquet(items_path).collect()
    if "item_id" not in items.columns:
        raise KeyError("items.parquet is missing item_id")
    return items

@dataclass(frozen=True)
class SplitResult:
    train: pl.DataFrame
    validation: pl.DataFrame
    test: pl.DataFrame
    items: pl.DataFrame

def load_splits(
    transaction_path: Path,
    items_path: Path,
    batch_size: int = 1_000_000,
) -> SplitResult:
    """Load and split transaction data by month."""
    print(f"Loading transactions from {transaction_path}...")
    raw = pl.scan_parquet(transaction_path)
    
    # Detect column names
    sample = raw.head(1).collect()
    print(f"Sample columns: {sample.columns}")
    customer_col, item_col, date_col, event_col, qty_col = _detect_columns(sample)
    print(f"  Detected: customer={customer_col}, item={item_col}, date={date_col}, event={event_col}, qty={qty_col}")
    
    # Parse dates and extract month
    raw = raw.with_columns(
        pl.col(date_col).cast(pl.Date).alias("date_parsed")
    ).with_columns(
        pl.col("date_parsed").dt.year().alias("year"),
        pl.col("date_parsed").dt.month().alias("month_number"),
    ).select([
        pl.col(customer_col).cast(pl.Int64).alias("customer_id"),
        pl.col(item_col).cast(pl.Utf8).alias("item_id"),
        pl.col("year"),
        pl.col("month_number"),
        pl.col(qty_col).cast(pl.Float32).alias("weight"),
        pl.col(event_col).alias("event_type"),
    ])
    
    # Filter for positive events and quantity > 0
    positive_types = {'purchase', 'buy', 'add_to_cart', 'click', 'view', 'bought', 'purchased'}
    base_filtered = raw.filter(
        (pl.col("year") == 2025) &
        (pl.col("event_type").str.to_lowercase().is_in(positive_types)) &
        (pl.col("weight") > 0)
    ).drop_nulls()
    
    # Define time split function
    def to_split_frame(months: Set[int]) -> pl.DataFrame:
        month_filters = [pl.col("month_number") == m for m in sorted(months)]
        if not month_filters:
            return pl.DataFrame({"customer_id": pl.Int64, "item_id": pl.Utf8, "weight": pl.Float32})
        combined_filter = month_filters[0]
        for month_filter in month_filters[1:]:
            combined_filter = combined_filter | month_filter
        return base_filtered.filter(combined_filter).group_by(
            "customer_id", "item_id"
        ).agg(pl.col("weight").sum()).collect()
    
    train_data = to_split_frame(TRAIN_MONTHS)
    val_data = to_split_frame(VAL_MONTHS)
    test_data = to_split_frame(TEST_MONTHS)
    items_data = load_items(items_path)
    
    print(f"✓ Data loaded: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")
    
    return SplitResult(train=train_data, validation=val_data, test=test_data, items=items_data)

def locate_data_files():
    """Find data files in Kaggle dataset."""
    if IS_KAGGLE:
        # List all available files recursively
        input_files = list(INPUT_DIR.glob('**/*.parquet'))
        print(f"Available Kaggle files:")
        for f in input_files:
            print(f"  - {f.name} (in {f.parent.name})")
        
        # Find transaction file (specific name only)
        trans_matches = list(INPUT_DIR.glob(f"**/{TRANSACTION_FILE}"))
        if not trans_matches:
            raise FileNotFoundError(f"Transaction file not found: {TRANSACTION_FILE}")
        trans_path = trans_matches[0]
        print(f"\n✓ Found transaction file: {trans_path.name}")
        
        # Find items file (try variants)
        items_path = None
        for filename in ITEMS_FILENAMES:
            matches = list(INPUT_DIR.glob(f"**/{filename}"))
            if matches:
                items_path = matches[0]
                print(f"✓ Found items file: {items_path.name}")
                break
        
        if not items_path:
            raise FileNotFoundError(f"Items file not found. Expected one of: {ITEMS_FILENAMES}")
        
        return trans_path, items_path
    else:
        # Local fallback
        trans_path = INPUT_DIR / TRANSACTION_FILE if INPUT_DIR.exists() else Path('data') / TRANSACTION_FILE
        items_path = INPUT_DIR / ITEMS_FILENAMES[0] if INPUT_DIR.exists() else Path('data') / ITEMS_FILENAMES[0]
        return trans_path, items_path

print("✓ Data loading functions defined")

✓ Data loading functions defined


## 5. Core Pipeline Functions

In [5]:
def _build_id_mappings(train: pl.DataFrame) -> Tuple[Dict[int, int], Dict[str, int], List[int], List[str]]:
    """Build bidirectional mappings between IDs and matrix indices."""
    users = sorted(train["customer_id"].unique().to_list())
    items = sorted(train["item_id"].unique().to_list())
    
    user_to_index = {uid: idx for idx, uid in enumerate(users)}
    item_to_index = {iid: idx for idx, iid in enumerate(items)}
    
    return user_to_index, item_to_index, users, items

def build_sparse_matrix(train: pl.DataFrame) -> Tuple[csr_matrix, Dict[int, int], Dict[str, int], List[int], List[str]]:
    """Build sparse CSR matrix from Polars DataFrame with memory optimization."""
    user_to_index, item_to_index, users, items = _build_id_mappings(train)
    
    # Extract arrays with optimized dtypes
    row_indices = np.array(
        [user_to_index[uid] for uid in train["customer_id"].to_list()],
        dtype=np.int32
    )
    col_indices = np.array(
        [item_to_index[iid] for iid in train["item_id"].to_list()],
        dtype=np.int32
    )
    values = np.ones(len(train), dtype=np.uint8)  # Binary implicit feedback
    
    # Build sparse matrix
    matrix = csr_matrix(
        (values, (row_indices, col_indices)),
        shape=(len(users), len(items)),
        dtype=np.float32
    )
    matrix.sum_duplicates()
    
    return matrix, user_to_index, item_to_index, users, items

class TruncatedSVDRecommender:
    """Collaborative filtering recommender using matrix factorization."""
    
    def __init__(self, n_components: int = 64, random_state: int = 42, n_iter: int = 100):
        self.n_components = n_components
        self.random_state = random_state
        self.model = TruncatedSVD(n_components=n_components, n_iter=n_iter, random_state=random_state)
        
        self.matrix = None
        self.user_to_index = None
        self.item_to_index = None
        self.index_to_item = None
        self.user_factors = None
        self.item_factors = None
        self.item_popularity = None
    
    def fit(self, train: pl.DataFrame) -> "TruncatedSVDRecommender":
        """Fit SVD model on interaction matrix."""
        matrix, user_to_index, item_to_index, _, items = build_sparse_matrix(train)
        self.matrix = matrix
        self.user_to_index = user_to_index
        self.item_to_index = item_to_index
        self.index_to_item = items
        self.model.fit(matrix)
        self.user_factors = self.model.transform(matrix)
        self.item_factors = self.model.components_.T
        self.item_popularity = np.asarray(matrix.sum(axis=0)).ravel().astype(float)
        return self
    
    def _fallback_ranking(self, top_k: int) -> List[str]:
        """Fallback to popularity-based ranking."""
        if self.item_popularity is None:
            return self.index_to_item[:top_k]
        top_indices = np.argsort(-self.item_popularity)[:top_k]
        return [self.index_to_item[i] for i in top_indices]
    
    def recommend(self, user_id: int, seen_items: Set[str] = None, top_k: int = 10) -> List[str]:
        """Generate recommendations for a user."""
        if seen_items is None:
            seen_items = set()
        
        if user_id not in self.user_to_index:
            return self._fallback_ranking(top_k)
        
        user_idx = self.user_to_index[user_id]
        user_vec = self.user_factors[user_idx]
        scores = user_vec @ self.item_factors.T
        
        # Mask seen items
        for item_id in seen_items:
            if item_id in self.item_to_index:
                item_idx = self.item_to_index[item_id]
                scores[item_idx] = -np.inf
        
        ranking = np.argsort(-scores)
        recommended = [self.index_to_item[index] for index in ranking[:top_k]]
        if len(recommended) < top_k:
            fallback = self._fallback_ranking(top_k)
            for item_id in fallback:
                if item_id not in recommended:
                    recommended.append(item_id)
                if len(recommended) == top_k:
                    break
        return recommended[:top_k]

def build_truth_map(frame: pl.DataFrame) -> Dict[int, Set[str]]:
    """Build ground truth map from validation/test data."""
    truth: Dict[int, Set[str]] = {}
    if frame.is_empty():
        return truth
    for row in frame.iter_rows(named=True):
        cid = int(row["customer_id"])
        iid = str(row["item_id"])
        if cid not in truth:
            truth[cid] = set()
        truth[cid].add(iid)
    return truth

def build_seen_map(frame: pl.DataFrame) -> Dict[int, Set[str]]:
    """Build seen items map from training data."""
    seen: Dict[int, Set[str]] = {}
    if frame.is_empty():
        return seen
    for row in frame.iter_rows(named=True):
        cid = int(row["customer_id"])
        iid = str(row["item_id"])
        if cid not in seen:
            seen[cid] = set()
        seen[cid].add(iid)
    return seen

print("✓ Core functions defined (matrix building, model, metrics)")

✓ Core functions defined (matrix building, model, metrics)


In [9]:
def precision_at_k(recommended: Sequence[str], truth: Set[str], top_k: int) -> float:
    """Compute Precision@K metric (PRIMARY METRIC for PIR)."""
    if top_k == 0:
        return 0.0
    hits = sum(1 for item_id in recommended[:top_k] if item_id in truth)
    return hits / top_k

def reciprocal_rank(recommended: Sequence[str], truth: Set[str]) -> float:
    """Compute Mean Reciprocal Rank."""
    for index, item_id in enumerate(recommended, start=1):
        if item_id in truth:
            return 1.0 / index
    return 0.0

def average_precision_at_k(recommended: Sequence[str], truth: Set[str], top_k: int) -> float:
    """Compute Average Precision@K."""
    if not truth:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for index, item_id in enumerate(recommended[:top_k], start=1):
        if item_id in truth:
            hits += 1
            precision_sum += hits / index
    denominator = min(len(truth), top_k)
    return precision_sum / denominator if denominator else 0.0

def intersection_over_union(recommended: Sequence[str], truth: Set[str], top_k: int) -> float:
    """Compute Intersection Over Union."""
    recommended_set = set(recommended[:top_k])
    intersection = len(recommended_set & truth)
    union = len(recommended_set | truth)
    return intersection / union if union else 0.0

def evaluate_recommender(
    model: TruncatedSVDRecommender,
    truth_map: Dict[int, Set[str]],
    seen_map: Dict[int, Set[str]],
    top_k: int = 10,
) -> Dict[str, float]:
    """Evaluate model on metrics.
    
    PRIMARY METRIC: Precision@K (most critical for real-world Kiosk deployment with limited display)
    SECONDARY METRICS: Total Correct Hits, IoU, MRR, MAP@K
    """
    users = sorted(truth_map.keys())
    if not users:
        return {"precision@k": 0.0, "total_hits": 0, "iou": 0.0, "mrr": 0.0, "map@k": 0.0, "evaluated_users": 0.0}
    
    precision_scores = []
    mrr_scores = []
    map_scores = []
    iou_scores = []
    total_hits = 0
    
    for user_id in users:
        truth = truth_map[user_id]
        seen = seen_map.get(user_id, set())
        recommendations = model.recommend(user_id, seen_items=seen, top_k=top_k)
        
        prec = precision_at_k(recommendations, truth, top_k)
        precision_scores.append(prec)
        mrr_scores.append(reciprocal_rank(recommendations, truth))
        map_scores.append(average_precision_at_k(recommendations, truth, top_k))
        iou_scores.append(intersection_over_union(recommendations, truth, top_k))
        total_hits += sum(1 for item_id in recommendations[:top_k] if item_id in truth)
    
    return {
        "precision@k": float(np.mean(precision_scores)),
        "total_hits": int(total_hits),
        "iou": float(np.mean(iou_scores)),
        "mrr": float(np.mean(mrr_scores)),
        "map@k": float(np.mean(map_scores)),
        "evaluated_users": float(len(users)),
    }

def popularity_baseline(train: pl.DataFrame, top_k: int = 10) -> List[str]:
    """Generate popularity-based recommendations."""
    if train.is_empty():
        return []
    ranking = (
        train.group_by("item_id", maintain_order=False)
        .agg(pl.col("weight").sum().alias("total_weight"))
        .sort("total_weight", descending=True)
        .head(top_k)
        .get_column("item_id")
        .to_list()
    )
    return [str(item) for item in ranking]

def make_submission(
    model: TruncatedSVDRecommender,
    customer_ids: Iterable[int],
    seen_map: Dict[int, Set[str]],
    top_k: int = 10,
) -> Dict[str, List[str]]:
    """Generate submission format."""
    submission: Dict[str, List[str]] = {}
    for customer_id in customer_ids:
        submission[str(customer_id)] = model.recommend(
            customer_id,
            seen_items=seen_map.get(customer_id, set()),
            top_k=top_k
        )
    return submission

def save_json_submission(submission: Dict[str, List[str]], output_path: Path) -> None:
    """Save submission as JSON."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(submission, ensure_ascii=False, indent=2), encoding="utf-8")

print("✓ Evaluation metrics defined (Precision@10 PRIMARY)")

✓ Evaluation metrics defined (Precision@10 PRIMARY)


## 6. Dry-Run Validation (Code Verification Only)

In [10]:
print("=" * 60)
print("DRY-RUN VALIDATION: Testing all functions with synthetic data")
print("=" * 60)

# Create minimal synthetic test data
print("\n1. Creating synthetic test data...")
train_synthetic = pl.DataFrame({
    "customer_id": [1, 1, 2, 2, 3, 3],
    "item_id": ["a", "b", "a", "c", "b", "d"],
    "weight": [1.0, 1.0, 1.0, 2.0, 1.0, 1.0],
})

val_synthetic = pl.DataFrame({
    "customer_id": [1, 2],
    "item_id": ["c", "b"],
    "weight": [1.0, 1.0]
})

print(f"   Train shape: {train_synthetic.shape}")
print(f"   Val shape: {val_synthetic.shape}")

# Test sparse matrix building
print("\n2. Testing sparse matrix construction...")
try:
    matrix, u2i, i2i, users, items = build_sparse_matrix(train_synthetic)
    print(f"   ✓ Matrix shape: {matrix.shape}")
    print(f"   ✓ Unique users: {len(users)}, Unique items: {len(items)}")
    print(f"   ✓ Matrix density: {matrix.nnz / (matrix.shape[0] * matrix.shape[1]):.2%}")
except Exception as e:
    print(f"   ✗ Error: {e}")

# Test model training
print("\n3. Testing model fitting...")
try:
    model = TruncatedSVDRecommender(n_components=2, random_state=42, n_iter=10)
    model.fit(train_synthetic)
    print(f"   ✓ Model fitted successfully")
    print(f"   ✓ User factors shape: {model.user_factors.shape}")
    print(f"   ✓ Item factors shape: {model.item_factors.shape}")
except Exception as e:
    print(f"   ✗ Error: {e}")

# Test truth/seen maps
print("\n4. Testing map builders...")
try:
    train_seen = build_seen_map(train_synthetic)
    val_truth = build_truth_map(val_synthetic)
    print(f"   ✓ Train seen map: {len(train_seen)} users")
    print(f"   ✓ Val truth map: {len(val_truth)} users")
except Exception as e:
    print(f"   ✗ Error: {e}")

# Test metrics
print("\n5. Testing metric functions...")
try:
    metrics = evaluate_recommender(model, val_truth, train_seen, top_k=2)
    print(f"   ✓ Metrics computed:")
    print(f"     ★ Precision@K (PRIMARY): {metrics['precision@k']:.4f}")
    print(f"     • Total Hits: {metrics['total_hits']}")
    print(f"     • IoU: {metrics['iou']:.4f}")
    print(f"     • MRR: {metrics['mrr']:.4f}")
    print(f"     • MAP@K: {metrics['map@k']:.4f}")
except Exception as e:
    print(f"   ✗ Error: {e}")

# Test recommendations
print("\n6. Testing recommendation generation...")
try:
    recs = model.recommend(1, seen_items=train_seen.get(1, set()), top_k=2)
    print(f"   ✓ Generated recommendations: {recs}")
except Exception as e:
    print(f"   ✗ Error: {e}")

# Test submission format
print("\n7. Testing submission format...")
try:
    submission = make_submission(model, [1, 2, 3], train_seen, top_k=2)
    print(f"   ✓ Submission created for {len(submission)} users")
    print(f"   ✓ Sample: {next(iter(submission.items()))}")
except Exception as e:
    print(f"   ✗ Error: {e}")

print("\n" + "=" * 60)
print("✓ DRY-RUN VALIDATION PASSED - All functions work correctly")
print("=" * 60)

DRY-RUN VALIDATION: Testing all functions with synthetic data

1. Creating synthetic test data...
   Train shape: (6, 3)
   Val shape: (2, 3)

2. Testing sparse matrix construction...
   ✓ Matrix shape: (3, 4)
   ✓ Unique users: 3, Unique items: 4
   ✓ Matrix density: 50.00%

3. Testing model fitting...
   ✓ Model fitted successfully
   ✓ User factors shape: (3, 2)
   ✓ Item factors shape: (4, 2)

4. Testing map builders...
   ✓ Train seen map: 3 users
   ✓ Val truth map: 2 users

5. Testing metric functions...
   ✓ Metrics computed:
     ★ Precision@K (PRIMARY): 0.5000
     • Total Hits: 2
     • IoU: 0.5000
     • MRR: 0.7500
     • MAP@K: 0.7500

6. Testing recommendation generation...
   ✓ Generated recommendations: ['d', 'c']

7. Testing submission format...
   ✓ Submission created for 3 users
   ✓ Sample: ('1', ['d', 'c'])

✓ DRY-RUN VALIDATION PASSED - All functions work correctly


## 7. Full Pipeline Execution (Ready to Run)

In [19]:
print("\n" + "=" * 60)
print("MAIN PIPELINE: Load data and build baseline")
print("=" * 60)

try:
    # Locate data files
    print("\nStep 1: Locating data files...")
    trans_path, items_path = locate_data_files()
    print(f"  Transaction file: {trans_path}")
    print(f"  Items file: {items_path}")
    
    # Load and split data
    print("\nStep 2: Loading and splitting data...")
    splits = load_splits(trans_path, items_path, batch_size=MODEL_CONFIG['batch_size'])
    print(f"  Train records: {len(splits.train)}")
    print(f"  Val records: {len(splits.validation)}")
    print(f"  Test records: {len(splits.test)}")
    print(f"  Item catalog: {len(splits.items)}")
    
    # Train model on validation set
    print("\nStep 3: Training model...")
    model = TruncatedSVDRecommender(
        n_components=MODEL_CONFIG['n_components'],
        random_state=MODEL_CONFIG['random_state'],
        n_iter=100
    )
    model.fit(splits.train)
    print(f"  ✓ Model trained on {len(splits.train)} interactions")
    
    # Build maps
    print("\nStep 4: Building evaluation maps...")
    train_seen_map = build_seen_map(splits.train)
    val_truth_map = build_truth_map(splits.validation)
    test_truth_map = build_truth_map(splits.test)
    print(f"  Train seen: {len(train_seen_map)} users")
    print(f"  Val truth: {len(val_truth_map)} users")
    print(f"  Test truth: {len(test_truth_map)} users")
    
    # Evaluate on validation
    print("\nStep 5: Evaluating on validation set...")
    val_metrics = evaluate_recommender(
        model,
        val_truth_map,
        train_seen_map,
        top_k=MODEL_CONFIG['top_k']
    )
    print(f"  ★ PRIMARY METRIC - Precision@10: {val_metrics['precision@k']:.4f}")
    print(f"     (Critical for Kiosk display space limitations)")
    print(f"  • Total Correct Hits: {val_metrics['total_hits']}")
    print(f"  • IoU: {val_metrics['iou']:.4f}")
    print(f"  • MRR: {val_metrics['mrr']:.4f}")
    print(f"  • MAP@K: {val_metrics['map@k']:.4f}")
    
    # Evaluate on test
    print("\nStep 6: Evaluating on test set...")
    test_metrics = evaluate_recommender(
        model,
        test_truth_map,
        train_seen_map,
        top_k=MODEL_CONFIG['top_k']
    )
    print(f"  ★ PRIMARY METRIC - Precision@10: {test_metrics['precision@k']:.4f}")
    print(f"     (Critical for Kiosk display space limitations)")
    print(f"  • Total Correct Hits: {test_metrics['total_hits']}")
    print(f"  • IoU: {test_metrics['iou']:.4f}")
    print(f"  • MRR: {test_metrics['mrr']:.4f}")
    print(f"  • MAP@K: {test_metrics['map@k']:.4f}")
    
    # Optional: Retrain on all data
    print("\nStep 7: Retraining on all data for submission...")
    full_data = pl.concat([splits.train, splits.validation, splits.test])
    print(f"  Total records for retraining: {len(full_data)}")
    final_model = TruncatedSVDRecommender(
        n_components=MODEL_CONFIG['n_components'],
        random_state=MODEL_CONFIG['random_state'],
        n_iter=100
    )
    final_model.fit(full_data)
    final_seen_map = build_seen_map(full_data)
    print(f"  ✓ Final model trained on {len(full_data)} interactions")
    
    # Generate submission
    print("\nStep 8: Generating submission...")
    all_customer_ids = set(final_model.user_to_index.keys())
    submission = make_submission(
        final_model,
        all_customer_ids,
        final_seen_map,
        top_k=MODEL_CONFIG['top_k']
    )
    print(f"  ✓ Submission created for {len(submission)} users")
    
    # Save submission
    print("\nStep 9: Saving submission...")
    output_file = OUTPUT_DIR / SUBMISSION_FILE
    save_json_submission(submission, output_file)
    print(f"  ✓ Saved to {output_file}")
    
    print("\n" + "=" * 60)
    print("✓ PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 60)
    
except FileNotFoundError as e:
    print(f"\n✗ Data files not found: {e}")
    print("\nTo run on Kaggle:")
    print("  1. Upload this notebook to Kaggle")
    print("  2. Add the PIR dataset as a data source")
    print("  3. Run all cells")
except Exception as e:
    print(f"\n✗ Pipeline error: {e}")
    import traceback
    traceback.print_exc()


MAIN PIPELINE: Load data and build baseline

Step 1: Locating data files...
  Transaction file: data\transaction_full_2025.parquet
  Items file: data\items.parquet

Step 2: Loading and splitting data...
Loading transactions from data\transaction_full_2025.parquet...

✗ Data files not found: The system cannot find the path specified. (os error 3): data/transaction_full_2025.parquet

To run on Kaggle:
  1. Upload this notebook to Kaggle
  2. Add the PIR dataset as a data source
  3. Run all cells


## Summary and Notes

### What This Notebook Does

1. **Collaborative Filtering Baseline**: Implements matrix factorization using TruncatedSVD on user-item interaction data
2. **Implicit Feedback**: Binary feedback (1 = purchase, 0 = no purchase) for efficient memory usage
3. **Time-Based Splits**: Separates data by month (Train: Jan-Sep, Val: Oct, Test: Nov 2025)
4. **Evaluation Metrics**: Computes Precision@K, MRR, MAP@K, and IoU on validation and test sets
5. **Kaggle-Ready**: Automatically detects Kaggle environment and uses appropriate paths

### Key Features

- **Polars-based data loading**: Lazy evaluation and efficient parquet I/O
- **Sparse matrix optimization**: Uses int32 indices, uint8 values for memory efficiency
- **Configurable hyperparameters**: Easy to adjust n_components, top_k, random_state
- **Dry-run validation**: Tests all code paths with synthetic data before running on real data
- **Error handling**: Graceful fallbacks if data files are missing (shows clear instructions)

### How to Use on Kaggle

1. Upload this notebook to a Kaggle Notebook
2. Add the PIR dataset(s) to the notebook (add data → select dataset)
3. Run all cells in order
4. The submission file will be saved to `/kaggle/working/submission.json`

### Local Development

For local testing:
- Ensure `transactions.parquet` and `items.parquet` are in the working directory
- Install dependencies: `pip install numpy polars scipy scikit-learn`
- Run cells individually or all at once

### Future Improvements

- Hyperparameter tuning via cross-validation
- Advanced architectures (neural collaborative filtering)
- Feature engineering (item metadata, user profiles)
- Ensemble methods combining multiple models
- Cold-start handling for new users/items